<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/02-visualizacao_resultados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1: Configuração e Criação da Pasta de Exportação
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from google.colab import drive
import os

# 1. Montagem do Drive
drive.mount('/content/drive')

# 2. Definição de Caminhos (AJUSTE O CAMINHO DO SEU BANCO ABAIXO)
DB_PATH = '/content/drive/MyDrive/mba-engsof-tcc/versao_final/base-dados.db'
EXPORT_PATH = '/content/drive/MyDrive/mba-engsof-tcc/versao_final/graficos-tcc'

# Cria a pasta de exportação se ela não existir
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📂 Pasta criada: {EXPORT_PATH}")

def get_connection():
    return sqlite3.connect(DB_PATH)

# Garante suporte a acentuação e visual limpo
sns.set_context("paper", font_scale=1.2)

# Configurações para qualidade ds imagens
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("✅ Ambiente configurado para exportação de imagens JPG.")

In [ ]:
# Célula 2: Geração de Nuvens de Palavras (Contraste Crise vs. Cura)
import nltk
from nltk.corpus import stopwords
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import os

nltk.download('stopwords', quiet=True)

def gerar_nuvens_contraste_existencial():
    conn = get_connection()

    # 1. Seleciona os eixos existenciais
    df_topicos = pd.read_sql_query("SELECT id, antidoto_referencia FROM topico WHERE id != 3", conn)

    # 2. Stopwords Refinadas (Foco em extrair a essência filosófica)
    stop_words_pt = set(stopwords.words('portuguese'))
    custom_stops = {
        'disse', 'então', 'veio', 'porque', 'pois', 'sobre', 'todos', 'tudo',
        'assim', 'ainda', 'outra', 'outros', 'será', 'pode', 'fazer', 'tão',
        'casa', 'filho', 'filhos', 'homem', 'mulher', 'terra', 'povo', 'rei',
        'senhor', 'deus', 'jesus', 'cristo', 'amém', 'ora', 'eis', 'vós',
        'teu', 'tua', 'meu', 'minha', 'toda', 'ano', 'anos', 'morreu', 'mortos',
        'israel', 'judá', 'jerusalém', 'egito', 'babilônia', 'filisteus', 'moisés', 'davi',
        'jacó', 'abraão', 'pedro', 'paulo', 'joão', 'senhor', 'deuses'
    }
    todas_stops = stop_words_pt.union(custom_stops)

    # 3. Mapeamento de Configuração Visual
    config_eixos = {
        0: {'mapa_cura': 'viridis', 'mapa_crise': 'magma', 'label': 'han'},
        1: {'mapa_cura': 'YlOrBr',  'mapa_crise': 'copper', 'label': 'bauman'},
        2: {'mapa_cura': 'coolwarm', 'mapa_crise': 'inferno', 'label': 'frankl'}
    }

    print("🎭 Iniciando geração de Nuvens de Palavras (Contraste Existencial)...")

    for _, row in df_topicos.iterrows():
        t_id = row['id']
        nome_eixo = row['antidoto_referencia']
        cfg = config_eixos.get(t_id)

        # Processamos os dois polos: Crise (-1) e Cura (1)
        for tipo_sentimento, label_sent in [(-1, 'crise'), (1, 'cura')]:
            query = f"""
                SELECT vl.texto_limpo
                FROM verso_limpo vl
                JOIN verso_topico vt ON vl.verso_id = vt.verso_id
                JOIN verso_sentimento s ON vl.verso_id = s.verso_id
                WHERE vt.topico_id = {t_id}
                AND s.sentimento_num = {tipo_sentimento}
            """
            df_textos = pd.read_sql_query(query, conn)

            if not df_textos.empty:
                # Normalização e Limpeza Final
                texto_final = " ".join(df_textos['texto_limpo'].fillna('').tolist()).lower()

                # Mapa de cores dinâmico
                cmap = cfg['mapa_cura'] if tipo_sentimento == 1 else cfg['mapa_crise']

                wordcloud = WordCloud(width=1600, height=900,
                                      background_color='white',
                                      max_words=50,
                                      min_word_length=3, # Filtra ruídos curtos
                                      stopwords=todas_stops,
                                      colormap=cmap,
                                      collocations=False,
                                      prefer_horizontal=0.9).generate(texto_final)

                plt.figure(figsize=(15, 8))
                plt.imshow(wordcloud, interpolation='bilinear')
                plt.axis('off')

                # Nome de arquivo estruturado para o TCC: nuvem_[autor]_[tipo].jpg
                file_name = f"nuvem_{cfg['label']}_{label_sent}.jpg"
                plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
                plt.close()
                print(f"✅ Gerada: {file_name} ({nome_eixo} - {label_sent.upper()})")
            else:
                print(f"⚠️ Amostra insuficiente para {nome_eixo} ({label_sent.upper()})")

    conn.close()
    print(f"\n✨ Processo concluído. Verifique os arquivos em: {EXPORT_PATH}")

gerar_nuvens_contraste_existencial()

In [ ]:
# Célula 3: Distribuição de ANTÍDOTOS (Positivos) por Gênero Literário
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_antidotos_por_genero_filtrado():
    conn = get_connection()

    # 1. Consulta SQL: Apenas Versos POSITIVOS (Antídotos) e Eixos Existenciais
    query = """
        SELECT
            g.nome as Genero,
            t.antidoto_referencia as Antidoto,
            COUNT(vt.verso_id) as Frequencia
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso v ON vt.verso_id = v.id
        JOIN livro l ON v.livro_id = l.id
        JOIN genero_literario g ON l.genero_id = g.id
        WHERE vs.sentimento_num = 1  -- FILTRO: Apenas a Cura (Positivo)
          AND t.id != 3              -- Exclui Narrativo/Outros
        GROUP BY Genero, Antidoto
    """

    df_dist = pd.read_sql_query(query, conn)
    conn.close()

    if df_dist.empty:
        print("⚠️ Dados não encontrados. Certifique-se de que a classificação e o sentimento foram processados.")
        return

    # 2. Pivotar e Garantir Ordem Fixa de Cores/Autores
    df_pivot = df_dist.pivot(index='Genero', columns='Antidoto', values='Frequencia').fillna(0)

    # --- REFINAMENTO: Ordem fixa para garantir integridade visual ---
    # Esta lista deve bater com a ordem das 'cores_harmonizadas' abaixo
    ordem_autores = [
        'Esgotamento (Han)',
        'Insignificância (Frankl)',
        'Transitoriedade (Bauman)'
    ]

    # Reordena as colunas apenas com as que de fato existem no DF
    colunas_presentes = [c for c in ordem_autores if c in df_pivot.columns]
    df_pivot = df_pivot[colunas_presentes]

    # Ordenar as linhas (Gêneros) pelo total para o efeito "escadinha"
    df_pivot['Total_Geral'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values(by='Total_Geral', ascending=True).drop(columns='Total_Geral')

    # 3. Configuração Estética
    sns.set_style("white")

    # Cores vinculadas à ordem: Verde (Han), Azul (Frankl), Laranja/Bronze (Bauman)
    cores_harmonizadas = ['#2E8B57', '#4682B4', '#D2691E']

    ax = df_pivot.plot(kind='barh',
                       stacked=True,
                       color=cores_harmonizadas,
                       figsize=(14, 8),
                       width=0.75,
                       edgecolor='white',
                       linewidth=1)

    # Rótulos e Títulos
    plt.xlabel('Volume de Antídotos (Versículos com Sentimento Positivo)', fontsize=12, labelpad=15)
    plt.ylabel('Gênero Literário', fontsize=12, labelpad=15)
    # plt.title('Volume de Antídotos Existenciais por Gênero Literário', fontsize=16, pad=20)

    # Adicionando os rótulos de dados (Data Labels) dentro das fatias
    for p in ax.patches:
        width = p.get_width()
        if width > 15: # Só exibe o número se houver espaço na fatia da barra
            ax.annotate(f'{int(width)}',
                        (p.get_x() + width / 2, p.get_y() + p.get_height() / 2),
                        ha='center', va='center',
                        fontsize=10, color='white', fontweight='bold')

    # Ajustes finais de layout
    sns.despine(left=False, bottom=False)
    plt.legend(title='Eixos Existenciais (Autores)',
               bbox_to_anchor=(1.02, 1),
               loc='upper left',
               frameon=False)

    plt.tight_layout()

    # 4. Exportação em alta resolução
    file_name = "distribuicao_antidotos_genero_final.jpg"
    save_path = os.path.join(EXPORT_PATH, file_name)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

    print(f"✅ Gráfico exportado com sucesso em: {save_path}")
    plt.show()

# Execução
gerar_grafico_antidotos_por_genero_filtrado()

In [ ]:
# Célula 4: Gauge de Resolutividade
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import unicodedata

# Silencia avisos para um log limpo
plt.rcParams.update({'figure.max_open_warning': 0})

def slugify(text):
    """Normaliza o nome para o sistema de arquivos."""
    text = text.split('(')[0].strip().lower()
    return "".join(c for c in unicodedata.normalize('NFKD', text)
                   if unicodedata.category(c) != 'Mn').replace(' ', '_')

def criar_gauge_puro_dados(valor, nome_arquivo, cor_eixo):
    # Escala de Sensibilidade (-0.25 a +0.25)
    min_escala, max_escala = -0.25, 0.25

    # Cálculo da posição do ponteiro
    valor_clamped = max(min_escala, min(max_escala, valor))
    posicao_graus = ((valor_clamped - min_escala) / (max_escala - min_escala)) * 180

    fig, ax = plt.subplots(figsize=(7, 3.5)) # Proporção ideal para semicírculo

    # 1. Arco de Fundo
    ax.pie([180, 180], colors=['#F0F0F0', 'white'], startangle=180,
           wedgeprops={'width': 0.3, 'edgecolor': 'white', 'linewidth': 2})

    # 2. Ponteiro Indicador
    ax.pie([posicao_graus - 1.5, 3, 180 - posicao_graus - 1.5, 180],
           colors=['none', cor_eixo, 'none', 'white'],
           startangle=180, counterclock=True,
           wedgeprops={'width': 0.35, 'edgecolor': 'none'})

    # 3. Valor Central (Único elemento de texto na imagem)
    # O sinal '+ ' ou '-' ajuda na interpretação rápida do saldo
    plt.text(0, 0.05, f"{valor:+.3f}", ha='center', va='center',
             fontsize=42, fontweight='bold', color='#2C3E50')

    # 4. Limpeza Absoluta
    ax.axis('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    sns.despine(left=True, bottom=True)

    # Exportação
    path_completo = os.path.join(EXPORT_PATH, nome_arquivo)
    plt.savefig(path_completo, dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f"✅ Gauge Gerado: {nome_arquivo} (Valor: {valor:+.4f})")

def processar_gauges():
    try:
        conn = get_connection()
        query = """
            SELECT
                t.id,
                t.antidoto_referencia,
                CAST(SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) -
                     SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) AS FLOAT) /
                     COUNT(vs.verso_id) as saldo_resolutividade
            FROM verso_topico vt
            JOIN topico t ON vt.topico_id = t.id
            JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
            WHERE t.id IN (0, 1, 2)
            GROUP BY t.id, t.antidoto_referencia
        """
        df = pd.read_sql_query(query, conn)
        conn.close()
    except Exception as e:
        print(f"❌ Erro: {e}")
        return

    # Cores fixas para identificação visual implícita
    cores_eixos = {0: '#2E8B57', 1: '#D2691E', 2: '#4682B4'}

    for _, row in df.iterrows():
        nome_arquivo = f"gauge_resolutividade_{slugify(row['antidoto_referencia'])}.jpg"

        criar_gauge_puro_dados(
            valor=row['saldo_resolutividade'],
            nome_arquivo=nome_arquivo,
            cor_eixo=cores_eixos.get(row['id'], '#333')
        )

processar_gauges()

In [ ]:
# Célula 5: Comparativo Crise vs. Cura por Gênero Literário (Versão Corrigida)
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_crise_vs_cura_robusto():
    conn = get_connection()

    # 1. Consulta SQL: Focada em garantir que todos os gêneros com dados apareçam
    # Usamos JOINs explícitos para garantir a integridade entre verso_id e verso_id
    query = """
        SELECT
            g.nome as Genero,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Cura_Positivo,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Crise_Negativo
        FROM genero_literario g
        JOIN livro l ON g.id = l.genero_id
        JOIN verso v ON l.id = v.livro_id
        JOIN verso_topico vt ON v.id = vt.verso_id
        JOIN verso_sentimento vs ON v.id = vs.verso_id
        WHERE vt.topico_id IN (0, 1, 2)
        GROUP BY g.nome
        HAVING (Cura_Positivo + Crise_Negativo) > 0
        ORDER BY (Cura_Positivo + Crise_Negativo) DESC
    """

    df_comp = pd.read_sql_query(query, conn)
    conn.close()

    if df_comp.empty:
        print("⚠️ A consulta não retornou dados. Verifique se as tabelas vt e vs estão povoadas.")
        return

    # 2. Preparação dos Dados (Melt)
    df_melted = df_comp.melt(id_vars='Genero', var_name='Tipo', value_name='Quantidade')
    df_melted['Tipo'] = df_melted['Tipo'].replace({
        'Cura_Positivo': 'Antídoto (Cura)',
        'Crise_Negativo': 'Problemática (Crise)'
    })

    # 3. Gráfico
    plt.figure(figsize=(14, 8))
    sns.set_style("whitegrid")

    # Cores acadêmicas: Verde para Cura, Coral para Crise
    paleta = {'Antídoto (Cura)': '#50C878', 'Problemática (Crise)': '#FF6B6B'}

    ax = sns.barplot(data=df_melted, x='Quantidade', y='Genero', hue='Tipo',
                     palette=paleta, edgecolor='white')

    # Ajustes estéticos
    plt.xlabel('Volume de Versículos (Frequência Absoluta)', fontsize=11)
    plt.ylabel('Gênero Literário', fontsize=11)
    plt.legend(title='Análise de Sentimento', loc='lower right', frameon=True)

    # Rótulos nas barras
    for p in ax.patches:
        val = int(p.get_width())
        if val > 0:
            ax.annotate(f'{val}',
                        (p.get_width(), p.get_y() + p.get_height() / 2),
                        ha='left', va='center', fontsize=9, xytext=(5, 0),
                        textcoords='offset points')

    sns.despine()
    plt.tight_layout()

    # 4. Salvamento
    file_name = "comparativo_crise_cura_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Gráfico salvo com sucesso: {file_name}")

gerar_grafico_crise_vs_cura_robusto()

In [ ]:
# Célula 6: Mapa de Calor de Resolutividade Existencial
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_heatmap_resolutividade():
    conn = get_connection()

    # SQL para calcular o Saldo Líquido (Pos - Neg) / Total por Gênero e Eixo
    query = """
        SELECT
            g.nome as Genero,
            t.antidoto_referencia as Autor,
            CAST(SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) -
                 SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) AS FLOAT) /
                 COUNT(vs.verso_id) as Saldo
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso v ON vt.verso_id = v.id
        JOIN livro l ON v.livro_id = l.id
        JOIN genero_literario g ON l.genero_id = g.id
        WHERE t.id IN (0, 1, 2)
        GROUP BY Genero, Autor
        HAVING COUNT(vs.verso_id) > 10
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    # Pivotar para o formato de matriz
    df_pivot = df.pivot(index='Autor', columns='Genero', values='Saldo').fillna(0)

    # Gráfico
    plt.figure(figsize=(14, 6))
    sns.heatmap(df_pivot, annot=True, cmap="RdYlGn", center=0, fmt=".3f",
                linewidths=.5, cbar_kws={'label': 'Índice de Resolutividade'})

    plt.title('Mapa de Calor: Eficácia do Antídoto por Gênero Literário', fontsize=14, pad=20)
    plt.xlabel('Gênero Literário', fontsize=12)
    plt.ylabel('Eixo Existencial (Autor)', fontsize=12)

    file_name = "heatmap_resolutividade_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Heatmap de Resolutividade salvo: {file_name}")

gerar_heatmap_resolutividade()

In [ ]:
# Célula 7: Distribuição de Densidade Emocional (Violin Plot)
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_violin_plot_existencial():
    conn = get_connection()

    query = """
        SELECT
            t.antidoto_referencia as Autor,
            vs.sentimento_num as Sentimento
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id IN (0, 1, 2)
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    plt.figure(figsize=(12, 7))
    sns.set_style("white")

    # Cores harmonizadas com o restante do trabalho
    paleta = {'Esgotamento (Han)': '#2E8B57', 'Insignificância (Frankl)': '#4682B4', 'Transitoriedade (Bauman)': '#D2691E'}

    # O Violin Plot mostra a densidade de versos em cada ponto da escala -1 a 1
    ax = sns.violinplot(data=df, x='Autor', y='Sentimento', hue='Autor',
                        palette=paleta, inner="quart", bw_adjust=.5, legend=False)

    plt.axhline(0, color='black', linestyle='--', alpha=0.3)
    plt.title('Densidade e Distribuição de Sentimentos por Eixo Existencial', fontsize=14, pad=20)
    plt.ylabel('Escala de Sentimento (-1: Crise | 0: Neutro | +1: Cura)', fontsize=11)
    plt.xlabel('Eixo Filosófico', fontsize=11)

    file_name = "densidade_emocional_violin.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Violin Plot salvo com sucesso: {file_name}")

gerar_violin_plot_existencial()

In [ ]:
# Célula 8: Workflow / etapas do processamento
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

def gerar_workflow_processamento():
    # Proporção ampla para garantir que as fontes maximizadas respirem
    fig, ax = plt.subplots(figsize=(24, 7), facecolor='white')
    ax.set_facecolor('white')

    etapas = [
        {"n": "1", "titulo": "Ingestão", "sub": "Business Und.", "libs": "google.colab\nos, gc", "tipo": "und"},
        {"n": "2", "titulo": "Ambiente", "sub": "Data Prep", "libs": "transformers\ntorch", "tipo": "prep"},
        {"n": "3", "titulo": "Carga", "sub": "Data Prep", "libs": "sqlite3\npandas", "tipo": "prep"},
        {"n": "4", "titulo": "Limpeza", "sub": "Data Prep", "libs": "re\nstring", "tipo": "prep"},
        {"n": "5", "titulo": "Tópicos", "sub": "Modeling", "libs": "bertopic\nsklearn, nltk", "tipo": "mod"},
        {"n": "6", "titulo": "Sentimento", "sub": "Modeling", "libs": "pysentimiento\ntqdm", "tipo": "mod"},
        {"n": "7", "titulo": "Avaliação", "sub": "Evaluation", "libs": "matplotlib\nseaborn, wordcloud", "tipo": "eval"}
    ]

    # Esquema de cores refinado para diferenciar Data Prep
    cores = {
        "und":  {"face": "#F5F5F5", "edge": "#9E9E9E", "text": "#424242", "lib_color": "#616161"}, # Cinza
        "prep": {"face": "#FFF3E0", "edge": "#FF9800", "text": "#E65100", "lib_color": "#EF6C00"}, # Laranja (Data Prep)
        "mod":  {"face": "#E3F2FD", "edge": "#1976D2", "text": "#0D47A1", "lib_color": "#1565C0"}, # Azul
        "eval": {"face": "#E8F5E9", "edge": "#388E3C", "text": "#1B5E20", "lib_color": "#2E7D32"}  # Verde
    }

    n_etapas = len(etapas)
    box_w, box_h = 1.25, 0.95
    espacamento = 1.65

    # Linha conectora de fundo
    ax.plot([0, (n_etapas-1) * espacamento], [0.5, 0.5], color='#F0F0F0',
            linewidth=15, zorder=1, solid_capstyle='round')

    for i, etapa in enumerate(etapas):
        x = i * espacamento
        y = 0.5
        estilo = cores[etapa["tipo"]]

        # 1. Box da Etapa
        rect = patches.FancyBboxPatch(
            (x - box_w/2, y - box_h/2), box_w, box_h,
            boxstyle="round,pad=0.04", linewidth=2.8,
            edgecolor=estilo["edge"], facecolor=estilo["face"], zorder=3
        )
        ax.add_patch(rect)

        # 2. Rótulo Etapa X
        ax.text(x - box_w/2, y + box_h/2 + 0.08, f"Etapa {etapa['n']}",
                fontsize=13, fontweight='bold', color='#757575', ha='left')

        # 3. Título Principal (Max)
        ax.text(x, y + 0.25, etapa["titulo"], ha='center', va='center',
                fontsize=18, fontweight='black', color=estilo["text"], zorder=4)

        # 4. Subtítulo CRISP-DM
        ax.text(x, y + 0.08, etapa["sub"], ha='center', va='center',
                fontsize=12, style='italic', color=estilo["text"], alpha=0.9, zorder=4)

        # 5. Bibliotecas Monospace
        ax.text(x, y - 0.22, etapa["libs"], ha='center', va='center',
                fontsize=12, fontweight='bold', color=estilo["lib_color"],
                family='monospace', zorder=4)

        # 6. Setas (Cores seguem a origem do fluxo)
        if i < n_etapas - 1:
            ax.annotate("", xy=(x + espacamento - box_w/2 - 0.06, y),
                        xytext=(x + box_w/2 + 0.06, y),
                        arrowprops=dict(arrowstyle='-|>', color=estilo["edge"],
                        lw=2.5, mutation_scale=25), zorder=2)

    ax.set_xlim(-1.0, (n_etapas - 1) * espacamento + 1.0)
    ax.set_ylim(-0.1, 1.1)
    ax.axis('off')

    plt.tight_layout()

    file_name = "workflow_processamento.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight', pad_inches=0.1)

    print(f"✅ Workflow com Data Prep destacado gerado: {file_name}")
    plt.show()

gerar_workflow_processamento()